# IFFT reconstruct and Match (ROreg with correct ROinv)

In [2]:
from pathlib import Path
import src.read_write as rw
import src.reconstruction as recon

# ---- Directories ----
dir_h5 = Path('data/raw')

dir_reconstruction = Path('data/all_reconstructions')
dir_reconstruction.mkdir(parents=True, exist_ok=True)

dir_b0_maps = Path('data/all_b0_maps')
dir_b0_maps.mkdir(parents=True, exist_ok=True)

dir_run_info = Path('data/all_run_info')
dir_run_info.mkdir(parents=True, exist_ok=True)

dir_idc = Path('data/image_domain_correction')
dir_idc.mkdir(parents=True, exist_ok=True)

dir_idc_results = Path('data/image_domain_correction/results')
dir_idc_results.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---- Fourier Transform CONTROL SCAN (ROreg_ES000) and INVERTED SCAN (ROinv_ES000) ----
for file in dir_h5.iterdir():

    if file.name.endswith(".h5"):
        if "ROreg_ES000" in file.name or "ROinv_ES000" in file.name:
            # ---- Load Data ----
            xml, kspace = rw.load_h5(file)

            # ---- Reconstruct Data ----
            img = recon.ifft(kspace)

            # ---- Export -----
            export_path = dir_idc_results / (file.stem + "_IFFT_OrderNA.nii.gz")
            rw.write_nii_from_xml(img, xml, export_path) # type: ignore
            print("File Exported: ", export_path)

# ---- Match IFFT reconstructions ----
list_reg = []
list_inv = []
for file in dir_idc_results.iterdir():

        if 'IFFT' in file.name and "ROreg_ES000" in file.name:
            list_reg.append(file)


for file in list_reg:
    inv_file = file.with_name(file.name.replace("ROreg", "ROinv"))
    if inv_file.exists():
        list_inv.append(inv_file)

print("\n Paired Lists:")
print(list_reg)
print(list_inv)

img shape :  (111, 111, 111)
Saved NIfTI file to data/image_domain_correction/results/position1_ROinv_ES000_IFFT_OrderNA.nii.gz
File Exported:  data/image_domain_correction/results/position1_ROinv_ES000_IFFT_OrderNA.nii.gz
img shape :  (111, 111, 111)
Saved NIfTI file to data/image_domain_correction/results/position2_ROinv_ES000_IFFT_OrderNA.nii.gz
File Exported:  data/image_domain_correction/results/position2_ROinv_ES000_IFFT_OrderNA.nii.gz
img shape :  (111, 111, 111)
Saved NIfTI file to data/image_domain_correction/results/position3_ROinv_ES000_IFFT_OrderNA.nii.gz
File Exported:  data/image_domain_correction/results/position3_ROinv_ES000_IFFT_OrderNA.nii.gz
img shape :  (111, 111, 111)
Saved NIfTI file to data/image_domain_correction/results/position3_ROreg_ES000_IFFT_OrderNA.nii.gz
File Exported:  data/image_domain_correction/results/position3_ROreg_ES000_IFFT_OrderNA.nii.gz
img shape :  (111, 111, 111)
Saved NIfTI file to data/image_domain_correction/results/position2_ROreg_ES000_

# Merge the matched IFFT nii Files

In [ ]:


list_merged = []
for i in range(len(list_reg)):

    position = list_reg[i].stem.split('_')[0]

    path_reg_ro_nii = list_reg[i]
    path_inv_ro_nii = list_inv[i]


    merged_nii_name = f'{position}--merged.nii.gz'

    merge_nii_path = dir_idc_results / merged_nii_name

    !fslmerge -t {merge_nii_path} {path_reg_ro_nii} {path_inv_ro_nii}

    list_merged.append(merge_nii_path)

    print(f'-> Niftis merged into: {merge_nii_path}\n\n')

Copied position3_ROreg_ES000_IFFT_OrderNA.nii.gz to data/image_domain_correction/results
Copied position3_ROinv_ES000_IFFT_OrderNA.nii.gz to data/image_domain_correction/results
-> Niftis merged into: data/image_domain_correction/results/position3--merged.nii.gz


Copied position1_ROreg_ES000_IFFT_OrderNA.nii.gz to data/image_domain_correction/results
Copied position1_ROinv_ES000_IFFT_OrderNA.nii.gz to data/image_domain_correction/results
-> Niftis merged into: data/image_domain_correction/results/position1--merged.nii.gz


Copied position2_ROreg_ES000_IFFT_OrderNA.nii.gz to data/image_domain_correction/results
Copied position2_ROinv_ES000_IFFT_OrderNA.nii.gz to data/image_domain_correction/results
-> Niftis merged into: data/image_domain_correction/results/position2--merged.nii.gz




# Calculate Deformation Field (warp)

In [ ]:
import time
import numpy as np

acqparams = f'{dir_idc}/acqparams.txt'
config = f'{dir_idc}/b02b0_1.cnf'


runtimes_all = []

for i in range(len(list_merged)):
    start = time.perf_counter()

    merge_nii = list_merged[i]
    position = merge_nii.stem.split('--')[0]

    !topup \
        --imain={merge_nii} \
        --datain={acqparams} \
        --config={config} \
        --out={dir_idc_results}/{position} \
        --dfout={dir_idc_results}/{position}_deformation_field_mm

    runtime = time.perf_counter() - start
    runtimes_all.append(runtime)

np.save(f'{dir_run_info}/runtimes_IDC.npy', np.array(runtimes_all))


In [9]:
print("Runtimes:")
for i in range(len(runtimes_all)):
    print(f'Position {i+1}: {runtimes_all[i]:.2f} seconds')


Runtimes:
Position 1: 398.61 seconds
Position 2: 402.30 seconds
Position 3: 403.29 seconds


# Apply Deformation Field

In [12]:
positions = ['position1', 'position2', 'position3']


for pos in positions:
    ifft_img = f'{dir_idc_results}/{pos}_ROreg_ES000_IFFT_OrderNA.nii.gz'
    warp_img = f'{dir_idc_results}/{pos}_deformation_field_mm_01.nii.gz'
    out_img = f'{dir_reconstruction}/{pos}_ROreg_ES000_IDC_OrderNA.nii.gz'
    !(applywarp \
        --in={ifft_img} \
        --ref={ifft_img} \
        --warp={warp_img} \
        --out={out_img} \
        --interp=spline)

    print(f"Warp {warp_img} applied to {ifft_img} \n Saved as {out_img}\n")

Warp data/image_domain_correction/results/position1_deformation_field_mm_01.nii.gz applied to data/image_domain_correction/results/position1_ROreg_ES000_IFFT_OrderNA.nii.gz 
 Saved as data/all_reconstructions/position1_ROreg_ES000_IDC_OrderNA.nii.gz

Warp data/image_domain_correction/results/position2_deformation_field_mm_01.nii.gz applied to data/image_domain_correction/results/position2_ROreg_ES000_IFFT_OrderNA.nii.gz 
 Saved as data/all_reconstructions/position2_ROreg_ES000_IDC_OrderNA.nii.gz

Warp data/image_domain_correction/results/position3_deformation_field_mm_01.nii.gz applied to data/image_domain_correction/results/position3_ROreg_ES000_IFFT_OrderNA.nii.gz 
 Saved as data/all_reconstructions/position3_ROreg_ES000_IDC_OrderNA.nii.gz



# Convert into Fieldmap in Hz ~data/b0_maps and Rename

In [ ]:
import numpy as np


for file in dir_idc_results.iterdir():
    if "deformation_field_mm_01" in file.name:


        position = file.stem.split('_')[0]
        b0_export_path = Path(f'{dir_b0_maps}/b0_map_{position}_IDC_OrderNA.npy')

        # ---- Load deformation field ----
        _ , def_field = rw.load_nii(file)

        # ---- Get Correct Orientation and Dimension ----
        d = -np.flip(def_field[..., 0], axis=0)

        # ---- Conversion mm -> Hz ----
        nr_readout_points = d.shape[0]
        dwell_time = 40*1e-06 # seconds
        readout_time = dwell_time * nr_readout_points
        voxel_size = 2 # mm

        b0 = d/(voxel_size * dwell_time * nr_readout_points)


        # ---- Export ----
        np.save(b0_export_path, b0)
        print(f"Saved IDC b0-map [Hz] as {b0_export_path}")


Saved IDC b0-map [Hz] as data/all_b0_maps/b0_map_position2_IDC_OrderNA.npy
Saved IDC b0-map [Hz] as data/all_b0_maps/b0_map_position3_IDC_OrderNA.npy
Saved IDC b0-map [Hz] as data/all_b0_maps/b0_map_position1_IDC_OrderNA.npy
